# Tri-ARC Kaggle Evaluation Notebook

This notebook bootstraps the offline Tri-ARC environment, executes the full ARC-AGI-2 evaluation split, and emits a competition-ready `submission.json`. Attach the following Kaggle datasets before running:
1. **tri-arc-code** – repository snapshot mounted at `/kaggle/input/tri-arc/Tri-ARC`.
2. **tri-arc-wheels** – prebuilt Python wheel cache located under `vendor/wheels/`.
3. **tri-arc-data** – ARC-AGI-2 datasets mounted alongside the repo (already referenced by `data/arcagi2/`).

All cells are idempotent; re-run individual stages as needed if the session times out.

## 1. Copy repository into the writable workspace

Kaggle mounts datasets as read-only. We copy the Tri-ARC repo (excluding the large Qwen checkpoint) into /kaggle/working/ for logs and outputs, then link models/Qwen2.5-Coder-7B-Instruct back to the read-only dataset under /kaggle/input/tri-arc/Tri-ARC/.


In [ ]:
import os
import shutil
from pathlib import Path

SOURCE = Path("/kaggle/input/tri-arc/Tri-ARC")
TARGET = Path("/kaggle/working/Tri-ARC")
MODEL_SUBDIR = Path("models/Qwen2.5-Coder-7B-Instruct")
READ_ONLY_MODEL = SOURCE / MODEL_SUBDIR
TARGET_MODEL = TARGET / MODEL_SUBDIR

if not SOURCE.exists():
    raise FileNotFoundError(f"Source repository missing: {SOURCE}")
if not READ_ONLY_MODEL.exists():
    raise FileNotFoundError(f"Read-only Qwen checkpoint missing: {READ_ONLY_MODEL}")

def ignore_large_payloads(current_dir, entries):
    current_path = Path(current_dir).resolve()
    model_parent = (SOURCE / "models").resolve()
    if current_path == model_parent:
        return ["Qwen2.5-Coder-7B-Instruct"]
    return []

def copy_repo_sans_models():
    if TARGET.exists():
        print("Tri-ARC repository already present in /kaggle/working.")
        return
    print("Copying repository into writable workspace (excluding Qwen weights)...")
    shutil.copytree(SOURCE, TARGET, ignore=ignore_large_payloads)
    print("Base repository copy complete.")

def ensure_model_symlink():
    TARGET_MODEL.parent.mkdir(parents=True, exist_ok=True)
    if TARGET_MODEL.is_symlink():
        if TARGET_MODEL.resolve() == READ_ONLY_MODEL.resolve():
            print("Model symlink already points to the read-only dataset.")
            return
        TARGET_MODEL.unlink()
    elif TARGET_MODEL.exists():
        if TARGET_MODEL.is_dir():
            shutil.rmtree(TARGET_MODEL)
        else:
            TARGET_MODEL.unlink()
    TARGET_MODEL.symlink_to(READ_ONLY_MODEL, target_is_directory=True)
    print(f"Linked {TARGET_MODEL} -> {READ_ONLY_MODEL}")

copy_repo_sans_models()
ensure_model_symlink()

print("Workspace ready:", TARGET)
print("Sample entries:", sorted(p.name for p in TARGET.iterdir())[:10])


## 2. Configure environment variables

Logs are redirected to `/kaggle/temp/` to avoid quota limits on `/kaggle/working/`. These variables propagate to the shell commands invoked later in the notebook.

In [ ]:
from pathlib import Path
import os

BASE_DIR = Path("/kaggle/working/Tri-ARC")
TEMP_LOGS = Path("/kaggle/temp/Tri-ARC/logs")
TEMP_LOGS.mkdir(parents=True, exist_ok=True)

READ_ONLY_MODEL = Path("/kaggle/input/tri-arc/Tri-ARC/models/Qwen2.5-Coder-7B-Instruct")
if not READ_ONLY_MODEL.exists():
    raise FileNotFoundError(f"Read-only Qwen checkpoint missing: {READ_ONLY_MODEL}")

os.environ["TRI_ARC_ROOT"] = str(BASE_DIR)
os.environ["TRI_ARC_DATA_ROOT"] = str(BASE_DIR / "data" / "arcagi2")
os.environ["TRI_ARC_CACHE_ROOT"] = str(BASE_DIR / "cache")
os.environ["TRI_ARC_LOG_ROOT"] = str(TEMP_LOGS)
os.environ["PYTHONPATH"] = str(BASE_DIR)
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TRI_ARC_RUN_ID"] = os.environ.get("TRI_ARC_RUN_ID", "kaggle_pass1")
os.environ["TRI_ARC_QWEN_MODEL_DIR"] = str(READ_ONLY_MODEL)

for key in [
    "TRI_ARC_ROOT",
    "TRI_ARC_DATA_ROOT",
    "TRI_ARC_LOG_ROOT",
    "CUDA_VISIBLE_DEVICES",
    "TRI_ARC_RUN_ID",
    "TRI_ARC_QWEN_MODEL_DIR",
]:
    print(f"{key} = {os.environ[key]}")


## 3. Bootstrap the Python environment

Set optional overrides (e.g., `TRI_ARC_ALLOW_PYPI=1` for online fallback or `TRI_ARC_PIP_EXTRA_INDEX_URL` for a custom CUDA index) before proceeding.
The next cell installs requirements directly inside the Kaggle runtime: it first consumes `/kaggle/working/Tri-ARC/vendor/wheels/` and only hits PyPI when fallback is explicitly allowed.

Upload the `tri-arc-wheels` dataset so the offline path succeeds; keep PyPI disabled for competition submissions unless the rules permit it.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("/kaggle/working/Tri-ARC")
WHEEL_ROOT = PROJECT_ROOT / "vendor" / "wheels"
REQUIREMENTS_FILE = PROJECT_ROOT / "requirements.txt"

ALLOW_PYPI = os.environ.get("TRI_ARC_ALLOW_PYPI", "0") == "1"
PIP_EXTRA_INDEX_URL = os.environ.get(
    "TRI_ARC_PIP_EXTRA_INDEX_URL",
    "https://download.pytorch.org/whl/cu121",
)

def run(cmd):
    print(f"[Tri-ARC] {' '.join(cmd)}")
    subprocess.check_call(cmd, cwd=PROJECT_ROOT)

offline_cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-index",
    "--find-links",
    str(WHEEL_ROOT),
    "-r",
    str(REQUIREMENTS_FILE),
]

offline_success = False
if WHEEL_ROOT.exists():
    try:
        run(offline_cmd)
        offline_success = True
    except subprocess.CalledProcessError as exc:
        print(f"[Tri-ARC] Offline install failed: {exc}")
else:
    print(f"[Tri-ARC] Wheel cache {WHEEL_ROOT} not found.")

if not offline_success:
    if not ALLOW_PYPI:
        raise RuntimeError(
            "[Tri-ARC] Offline install failed and TRI_ARC_ALLOW_PYPI!=1.",
        )
    print(
        f"[Tri-ARC] Falling back to PyPI (extra index: {PIP_EXTRA_INDEX_URL}).",
    )
    run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--extra-index-url",
            PIP_EXTRA_INDEX_URL,
            "-r",
            str(REQUIREMENTS_FILE),
        ]
    )

run([sys.executable, "--version"])
run([sys.executable, "-m", "pip", "--version"])

## 4. Run the full evaluation split

This cell launches the orchestrator with all solver modules enabled. Adjust `TRI_ARC_RUN_ID` if you need to differentiate multiple attempts.

In [ ]:
%%bash
set -euxo pipefail
cd /kaggle/working/Tri-ARC
source .venv/bin/activate
export CUDA_VISIBLE_DEVICES=${CUDA_VISIBLE_DEVICES:-0,1,2,3}
RUN_ID=${TRI_ARC_RUN_ID:-kaggle_pass1}
python scripts/run_eval.py --preset evaluation --split evaluation --run-id ${RUN_ID} --log-level INFO

## 5. Inspect run artifacts

Quick sanity check to confirm that metrics, runtime logs, and summaries exist before producing the submission file.

In [ ]:
import json
import os
from pathlib import Path

run_id = os.environ.get("TRI_ARC_RUN_ID", "kaggle_pass1")
run_root = Path("/kaggle/working/Tri-ARC/reports/eval_runs/full") / run_id
summary_path = run_root / "summary.json"
metrics_path = run_root / "metrics.jsonl"
runtime_log = run_root / "logs" / "runtime.log"

print("Run directory:", run_root)
for path in [summary_path, metrics_path, runtime_log]:
    print(f"{path}: {'OK' if path.exists() else 'MISSING'}")

if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2)[:1000])

## 6. Build `submission.json`

The helper script consolidates `outputs/pass2.json` into the two-attempt-per-output format required by Kaggle.

In [ ]:
%%bash
set -euxo pipefail
cd /kaggle/working/Tri-ARC
source .venv/bin/activate
python tools/make_submission.py --output submission.json
ls -lh submission.json outputs/pass2.json

## 7. Package artifacts for download / dataset upload

Bundle the key outputs (submission file, pass2 archive, logs, and reports) into a single tarball. Use the Kaggle “Download Output” button to retrieve it once the notebook finishes.

In [ ]:
%%bash
set -euxo pipefail
cd /kaggle/working/Tri-ARC
RUN_ID=${TRI_ARC_RUN_ID:-kaggle_pass1}
tar -czf tri_arc_artifacts.tar.gz \
  submission.json \
  outputs/pass2.json \
  reports/eval_runs/full/${RUN_ID}
ls -lh tri_arc_artifacts.tar.gz

In [ ]:
import subprocess
import tarfile
from pathlib import Path

REPO_DIR = Path("/kaggle/working/Tri-ARC")
if not (REPO_DIR / ".git").exists():
    raise FileNotFoundError(f"Git repository not found at {REPO_DIR}")

result = subprocess.run(
    ["git", "status", "--porcelain=1", "--untracked-files=all"],
    cwd=REPO_DIR,
    text=True,
    capture_output=True,
    check=True,
)

paths = []
for line in result.stdout.splitlines():
    if not line.strip():
        continue
    entry = line[3:]
    entry = entry.strip()
    if entry.startswith('"') and entry.endswith('"'):
        entry = entry[1:-1]
    paths.append(entry)

paths = sorted(dict.fromkeys(paths))
if not paths:
    print("No changed or untracked files detected; nothing to archive.")
else:
    archive = REPO_DIR / "tri_arc_changes_only.tar.gz"
    with tarfile.open(archive, "w:gz") as tar:
        for rel in paths:
            target = REPO_DIR / rel
            if target.exists():
                tar.add(target, arcname=rel)
            else:
                print(f"Skipping missing path {rel}")
    print(f"Archived {len(paths)} paths to {archive}")
    for rel in paths[:20]:
        print(" -", rel)
    size_kib = archive.stat().st_size / 1024
    print(f"Archive size: {size_kib:.1f} KiB")